In [1]:
import pandas as pd
import numpy as np
from collections import Counter
import re
import spacy
import nltk
from nltk.corpus import words, stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import contractions
from tqdm import tqdm
tqdm.pandas()

In [2]:
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('stopwords', quiet=True)

True

In [3]:
ENGLISH_WORDS = set(w.lower() for w in words.words())
STOP_WORDS = set(w for w in stopwords.words('english'))

# Load & Inspect Data

In [4]:
DATA_PATH = 'data/fakeNewsClassificationData.csv'

In [5]:
raw_data = pd.read_csv(DATA_PATH)

In [6]:
display(raw_data.head())

,title,author,text,label
0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...,1
1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...,0
2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ...",1
3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...,1
4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...,1


In [7]:
print("Size of dataset:", raw_data.shape[0])

Size of dataset: 20800


## Sample Record

In [8]:
for c in raw_data.columns:
    print(c, ":", raw_data.loc[5][c])
    print()

title : Jackie Mason: Hollywood Would Love Trump if He Bombed North Korea over Lack of Trans Bathrooms (Exclusive Video) - Breitbart

author : Daniel Nussbaum

text : In these trying times, Jackie Mason is the Voice of Reason. [In this week’s exclusive clip for Breitbart News, Jackie discusses the looming threat of North Korea, and explains how President Donald Trump could win the support of the Hollywood left if the U. S. needs to strike first.  “If he decides to bomb them, the whole country will be behind him, because everybody will realize he had no choice and that was the only thing to do,” Jackie says. “Except the Hollywood left. They’ll get nauseous. ” “[Trump] could win the left over, they’ll fall in love with him in a minute. If he bombed them for a better reason,” Jackie explains. “Like if they have no transgender toilets. ” Jackie also says it’s no surprise that Hollywood celebrities didn’t support Trump’s strike on a Syrian airfield this month. “They were infuriated,” he say

## Handle Missing Data

In [9]:
print("Count of records with missing data:")
print(raw_data.isna().sum())

Count of records with missing data:
title      558
author    1957
text        39
label        0
dtype: int64


In [10]:
prepped_data = raw_data.dropna(subset=['text'])
print(prepped_data.isna().sum())

title      558
author    1918
text         0
label        0
dtype: int64


## Handle Duplicate Data

In [11]:
prepped_data.duplicated().sum()

109

In [12]:
prepped_data = prepped_data.drop_duplicates()

In [13]:
prepped_data.shape

(20652, 4)

## Check for Meta-data Leakage

In [14]:
for idx in list(prepped_data['title'][prepped_data['title'].isna()].index):
    print(prepped_data.loc[idx])
    print()

title                                                   NaN
author                                          Dairy✓ᵀᴿᵁᴹᴾ
text      Sounds like he has our president pegged. What ...
label                                                     1
Name: 53, dtype: object

title                                                   NaN
author                                            Anonymous
text      Same people all the time , i dont know how you...
label                                                     1
Name: 120, dtype: object

title                                                   NaN
author                                    SeekSearchDestory
text      You know, outside of any morality arguments, i...
label                                                     1
Name: 124, dtype: object

title                                                   NaN
author                                            Anonymous
text      There is a lot more than meets the eye to this...
label                  

In [15]:
prepped_data['label'][prepped_data['title'].isna()].unique()

array([1], dtype=int64)

* It appears that all the records with missing title happen to be fake news. So, the absence of a title could be a characterisitic of fake news.

In [16]:
c_fake = 0
c_real = 0
for i,(author,title,label) in enumerate(prepped_data[['author','title','label']].values):
    if 'anonymous' in str(title).lower():
        print(author, "->", title, "->", label)
        if label==1:
            c_fake += 1
        else:
            c_real += 1

Starkman -> Anonymous Donor Pays $2.5 Million To Release Everyone Arrested At The Dakota Access Pipeline -> 1
Dave Hodges -> Scandalous Video Footage From Anonymous Exposes Huma and Hillary -> 1
wmw_admin -> Anonymous Release Bone Chilling Video of Huma Abedin that Every American Needs to See -> 1
Nick Hallett -> Anonymous American Donor Funds ’Some Girls Have Penises, Some Boys Vaginas’ Poster Campaign -> 0
Anonymous: World War 3 Is On The Horizon (In 2016) – Collective Evolution -> Comment on US Congressperson Says It’s Time For America To Disclose The Existence of Extraterrestrials by Anonymous: World War 3 Is On The Horizon (In 2016) – Collective Evolution -> 1
righteous -> What is Anonymous? -> 1
World War 3 News 2016: Anonymous Declares WW3 'on the Horizon' -> Comment on Anonymous: World War 3 Is On The Horizon (In 2016) by World War 3 News 2016: Anonymous Declares WW3 'on the Horizon' -> 1
Dave Hodges -> Anonymous: America’s Last Hope-You Have Been Warned -> 1
Alex Cooper -> Ano

In [17]:
c_fake, c_real

(27, 3)

* Majority of the records in which the title contained the word 'anonymous', the news turned out to be fake. However, the number is very small compared to the total number of records.

In [18]:
c_fake = 0
c_real = 0
for i,(author,title,label) in enumerate(prepped_data[['author','title','label']].values):
    if re.match(r'\bno\b|\bnone\b|\bnil\b', str(title).lower()):
        print(author, "->", title, "->", label)
        if label==1:
            c_fake += 1
        else:
            c_real += 1

Remy Porter -> No Account for You -> 1
Guest Author -> No, Hate Crimes Have NOT ‘Intensified’ Since Trump’s Election -> 1
Activist Post -> No Matter Who Wins: The Case Against Hillary Clinton Is Alive -> 1
John Carney -> No Americans Need Apply: U.S. Government Twice as Open to Foreign Business as Five Top Trading Partners Put Together -> 0
Dennis Overbye -> No Escape From Black Holes? Stephen Hawking Points to a Possible Exit - The New York Times -> 0
Jack Healy -> No Health Insurance Is Hard. No Phone? Unthinkable. - The New York Times -> 0
Dylan Gwinn -> No Prison Time for White Football Player Accused of Raping Mentally Disabled Black Teen With Coat Hanger - Breitbart -> 0
Dylan Gwinn -> No Monkeying Around: NFL Network’s Mayock Threatens to Walk Off Set After Orangutan Announces Draft Pick - Breitbart -> 0
Amanda Hess -> None of Us Are Safe From Getting ‘Owned’ - The New York Times -> 0
# 1 NWO Hatr -> No-Fly Zone Declared as Militarized Police Prep for Assault on ‘Front-Line Camp

In [19]:
c_fake, c_real

(21, 21)

In [20]:
prepped_data['label'][prepped_data['author'].isna()].value_counts()

label
1    1867
0      26
Name: count, dtype: int64

* Majority of the records with missing author name are fake, while a small number of them are real too. So it cannot be a definitive trait.

In [21]:
idxs = []
for i,author in enumerate(prepped_data['author'].values):
    if 'anonymous' in str(author).lower() or  re.match(r'\bno\b|\bnone\b|\bnil\b', str(author).lower()):
        print(author)
        idxs.append(i)

Anonymous Coward (UID 12781064)
Anonymous
Anonymous
Anonymous Coward (UID 11897093)
Anonymous
Anonymous Coward (UID 73268493)
Anonymous
Anonymous
Anonymous Coward (UID 19747754)
Anonymous
Anonymous Coward (UID 26968733)
No Author
Anonymous Coward (UID 47910020)
Anonymous Coward (UID 73270245)
Anonymous Coward (UID 73268493)
Anonymous
Anonymous
No Author
Anonymous
Anonymous Coward (UID 73270427)
No Author
Anonymous: World War 3 Is On The Horizon (In 2016) – Collective Evolution
No Author
Anonymous
Anonymous
Anonymous
No Author
Anonymous Coward (UID 58307359)
Anonymous
Anonymous Coward (UID 73270620)
World War 3 News 2016: Anonymous Declares WW3 'on the Horizon'
Anonymous
Anonymous
Anonymous Activist
Anonymous
Anonymous Coward (UID 18807137)
Anonymous Coward (UID 73271258)
Anonymous
Anonymous
Anonymous Coward (UID 73270906)
Anonymous
No Author
Anonymous
No Author
Anonymous
No Author
Anonymous
Anonymous: There Is No One Way To Live On This Planet, But We Can Be Harmonious – Collective Evo

In [22]:
print(len(idxs))

116


In [23]:
prepped_data.iloc[idxs]['label'].unique()

array([1], dtype=int64)

* Other than explicitly missing values for author column, there are records with names that contain the word 'anonymous' and 'no author', these have been found to be all fake.

In [30]:
def metadata_stats(df, col):
    # col becomes index
    stats = df.groupby(col).agg(
        total=('text', 'size'),
        fakeness=('label', 'mean'))
    # changing col back to a column and sorting groupby table
    stats = stats.reset_index().sort_values('total', ascending=False)
    return stats

In [31]:
author_stats = metadata_stats(prepped_data, 'author')

In [32]:
author_stats.head(20)

,author,total,fakeness
2944,Pam Key,243,0.004115
3929,admin,192,1.000000
1762,Jerome Hudson,166,0.000000
724,Charlie Spiering,141,0.000000
1857,John Hayward,140,0.000000
2090,Katherine Rodriguez,124,0.000000
3845,Warner Todd Huston,122,0.000000
1520,Ian Hanchett,119,0.000000
577,Breitbart News,118,0.000000
914,Daniel Nussbaum,112,0.000000


In [35]:
author_leakage = author_stats[(author_stats['total']>50) & ((author_stats['fakeness']<0.05) | (author_stats['fakeness']>0.95))]
print("Authors who are mostly found for one class:")
display(author_leakage)

Authors who are mostly found for one class:


,author,total,fakeness
2944,Pam Key,243,0.004115
3929,admin,192,1.000000
1762,Jerome Hudson,166,0.000000
724,Charlie Spiering,141,0.000000
1857,John Hayward,140,0.000000
2090,Katherine Rodriguez,124,0.000000
3845,Warner Todd Huston,122,0.000000
1520,Ian Hanchett,119,0.000000
577,Breitbart News,118,0.000000
914,Daniel Nussbaum,112,0.000000


* We can see that there are several authors corresponding to whom there are either 100% fake news or 100% real news. So, if the author data is fed to a classifier along with the text, the mdoel may simply learn based on the author names instead of actual text and language features. Since, clearly, there is leakage from the author data, we will not be including it for further processing.

In [44]:
title_stats = metadata_stats(prepped_data, 'title')
title_stats.head(10)

,title,total,fakeness
15400,The Dark Agenda Behind Globalism And Open Borders,5,1.0
6764,Get Ready For Civil Unrest: Survey Finds That ...,5,1.0
9609,Let’s Be Clear – A Vote For Warmonger Hillary ...,4,1.0
12658,Public vs. Media on War,4,1.0
15490,The Fix Is In: NBC Affiliate Accidentally Post...,4,1.0
10400,Michael Moore Owes Me $4.99,4,1.0
18350,What to Cook This Week - The New York Times,4,0.0
9469,Las imágenes libres de derechos más destacadas...,4,1.0
15833,The U.S. National Bird Is Now a Drone,4,1.0
18750,Will Barack Obama Delay Or Suspend The Electio...,4,1.0


In [46]:
title_stats.tail(10)

,title,total,fakeness
6634,"Gary Johnson Goes Zombie, Tries to Bite Reporters",1,1.0
6633,Gary Johnson Equates Syria Deaths Caused by As...,1,0.0
6632,Gary Cohn Relaunches War on Coal: Fuel from Am...,1,0.0
6631,"Garry Marshall, ‘Pretty Woman’ Director, Dies ...",1,0.0
6630,Garrison Keillor Turns Out the Lights on Lake ...,1,0.0
6629,Garlic: 12 Serious Health Benefits | Undergrou...,1,1.0
6628,Garlic Beats Drug In Detoxifying Lead Safely F...,1,1.0
6627,Gardaí Strike Negotiations Get Off To Bad Star...,1,1.0
6626,Gardasil is a Decision We Will Always Regret,1,1.0
19763,🚨Bill Clinton and Hillary Lolita Express Pedop...,1,1.0


In [50]:
title_stats[['total','fakeness']].value_counts()

total  fakeness
1      0.0         10377
       1.0          9078
2      1.0           261
3      1.0            31
4      1.0            11
2      0.0             3
5      1.0             2
4      0.0             1
Name: count, dtype: int64

* Majority of the titles are unique. The few that are duplicates are of records belonging to the same class, but the numbers aren't significant to allow the model to cheat. Hence, we will retain the titles and concatenate them to the text.

In [53]:
prepped_data.drop(columns='author', inplace=True)

In [57]:
prepped_data['full_text'] = prepped_data['title'].fillna("") + " " + prepped_data['text']

In [58]:
prepped_data.head()

,title,text,label,full_text
0,House Dem Aide: We Didn’t Even See Comey’s Let...,House Dem Aide: We Didn’t Even See Comey’s Let...,1,House Dem Aide: We Didn’t Even See Comey’s Let...
1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Ever get the feeling your life circles the rou...,0,"FLYNN: Hillary Clinton, Big Woman on Campus - ..."
2,Why the Truth Might Get You Fired,"Why the Truth Might Get You Fired October 29, ...",1,Why the Truth Might Get You Fired Why the Trut...
3,15 Civilians Killed In Single US Airstrike Hav...,Videos 15 Civilians Killed In Single US Airstr...,1,15 Civilians Killed In Single US Airstrike Hav...
4,Iranian woman jailed for fictional unpublished...,Print \nAn Iranian woman has been sentenced to...,1,Iranian woman jailed for fictional unpublished...


## Text Length Statistics

In [59]:
prepped_data['char_count'] = prepped_data['full_text'].astype(str).map(len)

In [60]:
prepped_data['char_count'].describe()

count     20652.000000
mean       4628.469543
std        5129.856542
min           2.000000
25%        1712.750000
50%        3444.500000
75%        6359.000000
max      143035.000000
Name: char_count, dtype: float64

In [62]:
print("Shortest Text:")
print(prepped_data.loc[prepped_data['char_count'].idxmin(), ['text', 'char_count','label']])

Shortest Text:
text          f
char_count    2
label         1
Name: 786, dtype: object


In [63]:
print("Longest Text:")
print(prepped_data.loc[prepped_data['char_count'].idxmax(), ['text', 'char_count','label']])

Longest Text:
text          Заседание Международного дискуссионного клуба ...
char_count                                               143035
label                                                         1
Name: 19764, dtype: object


# Preprocess Data

We will create new columns:
1. 'text_clean': lowercased, urls removed, contractions fixed, punctuations removed
2. 'tokens': list of lemmatized tokens after removing stopwords and non-english words
3. 'text_lemmatized': tokens joined back after lemmatization for TF-IDF
4. 'token_count'


We will makes these for both text and full_text.

In [64]:
def fix_contractions_safe(s):
  try:
    return contractions.fix(s)
  except:
    return s
def clean_text(s):
    s = str(s)
    print(s[:10])
    # normalizing quotations
    quote_maps = {'“':'"', '”':'"', "‘":"'", "’":"'"}
    for curly_quote,quote in quote_maps.items():
        s = s.replace(curly_quote, quote)
    # remove non-ASCII
    s = re.sub(r'[^\x00-\x7F]+', ' ', s)
    # fixing contractions
    s = fix_contractions_safe(s)
    # removing urls
    s = re.sub(r'(https?:\/\/)?([\w\.-]+)\.([A-Za-z\.]{2,6})([\/\w \.-]*)*\/?', ' ', s)
    # removing punctuations (all non-alphanumeric characters and non-whitespaces)
    s = re.sub(r'[^a-zA-Z0-9 ]', ' ', s)
    # removing unnecessary whitespaces, newlines, tabspaces, etc.
    s = re.sub(r'\s+', ' ', s).strip()
    # returning lowercase text
    return s.lower()

In [65]:
prepped_data['text_clean'] = prepped_data['text'].progress_map(clean_text)

  1%|▍                                                                            | 112/20652 [00:00<00:35, 578.45it/s]

House Dem 
Ever get t
Why the Tr
Videos 15 
Print 
An 
In these t
Ever wonde
PARIS  —  
Donald J. 
A week bef
Organizing
The BBC pr
The myster
Clinton Ca
Yes, There
Guillermo 
The scanda
A Caddo Na
FBI Closes
Wednesday 
Email 
Sin
Screenwrit
Sunday on 
Massachuse
Orders for
Email 
In 
JERUSALEM 
Humiliated
Andrea Tan
Hillary Cl
During a d
Country: I
LONDON  — 
MIDLAND, T
Don Halcom
VOROKHOBIN
Why We Are
Open Threa
Rep. Luis 
This artic
A newly de
  Now, num
The fallou
Can I have
A group of
Executives
By Gordon 
0 коммента
SAN FRANCI
“The publi
0 0 With h
Dueling Dr
A new stud
Sounds lik
Samantha B
The Trump 
Click Here
1 Shares
1
If Donald 
Editors’ n
The United
I wonder w
Three pion
17 mins ag
The first 
WASHINGTON
Oregon Liv
Clinton Ca
WASHINGTON
— Bernie S
  22, 2016
In Hillary
What does 
NEW ORLEAN
HANGZHOU, 
The Upshot
Email 
Get

Bob Unruh
WASHINGTON
Sunday on 
(Want to g
(AP)  —   
 
0 comments
With an ex
Mary Tyler
By Dr. Mar
DENVER  — 
Share on T
The fictio
This summe
It is no

  1%|▋                                                                            | 180/20652 [00:00<00:32, 623.52it/s]

A high sch
BREAKING :
Snapchat i
Zero Hedge
After a st
DALLAS  — 
OTTAWA  — 
Marital st
WASHINGTON
Same peopl

This arti
“Chapo Tra
HONG KONG 
You know, 
The Mother
By wmw_adm
GREENBELT,
KABUL, Afg
News ancho
House Mino
By Pao Cha
I spent la
ASTON, Pa.
Politics I

21st Cent
Democrats 
Editor’s N
- Advertis
RIO DE JAN
There is a
(Before It
  U.N. Sec
Google Pin

Crushing 
Serena Shi
Illegal al

Geert Wil
Four Israe
Business I
Comments 

Despite st
Normani Ko
WASHINGTON
WASHINGTON
The Muslim
WASHINGTON
The Secret
Tweet Widg
It might b
Think the 
Fluoridati
Lt. Gen. J
Comments 

The questi
TOKYO  —  
BIRMINGHAM
Police in 
 
Donate Rem
Sunday on 
‹ › Arnald
  Guest   
As an exam
Sunday, 30
A wave of 
Comments 

Carey Wedl
Teacher un
CAIRO  —  
Normally, 
They are m
BEIRUT, Le
WHO cancer
Stuck at y
#FROMTHEFR
Steve Harv
U. S servi
UK citizen
After Vets
‘Have you 
1 comment 
20 Foods T
Death of t
Wed, 26 Oc
They got t
Home / Bad
By Ann Des
   Donald 
MIAMI BEAC
The medica
This Ameri

A lot o

  2%|█▍                                                                           | 381/20652 [00:00<00:31, 652.20it/s]

...

WASHINGTON
WARSAW  — 
Los direct
Waking Tim
Two Indy 5
 
Photos rel
DUBLIN  — 

This arti
Hillary Ar
-Onions -R
It is know
(AP) CHICA
Welcome to
Huckabee R
He survive
As flying 
South Afri
Posted on 
Wednesday 
Print 
Of 
November 8
Better thi
175 Views 
A leading 
Two Boston
The accuse
 
He is aliv
NICOSIA, C
On her Feb
I’m on the
By LAURA M
Lena Dunha
President 
BREAKING: 
The Times 
Go to Arti
Thu, 27 Oc
Seven of t
BNI Store 
How do you
Democrats 
Home › POL
by Lambert
AUSTIN, Te
This weeke
■   Donald
Trey Yings
LAS VEGAS 
  
He call
RIO DE JAN
Freitag, 4
Documentar
The corpor
President 
The USA Er
Half a cen
President 
[Ed. – Way
Pete Souza
Israeli of
Nearly 150
NAIROBI, K
We Are Cha
Woman 'eat
ROME  —   
Selected N
WASHINGTON
Donald J. 
by Vladimi
Iran will 
Pinterest 
PACHINKOBy
It is gett
Donald Tru
(Want to g
Email 
The
JERUSALEM 
CANNES, Fr
WASHINGTON
Slovenia A
By Jon Rap
  Von wege
PEWAUKEE, 
The Obamac
WASHINGTON
THE END GA
in: False 
Dallas pol
Whether th
Email 
The
LOS AN

 99%|██████████████████████████████████████████████████████████████████████████▏| 20432/20652 [00:25<00:00, 787.43it/s]

Asteroid W
Thursday o
Recalling 
7 hours ag
WASHINGTON
(Before It
Of course,
Can’t make
Carolyn B.
DETROIT  —
People ove
[Graphic: 
Tweet 
Ped
WASHINGTON
MELBOURNE,
There’s no
WASHINGTON
Why Was A 
  
Trump’s
Home | Wor
On Monday,
One of the
As more me
posted by 
While in t
27444 SHAR
in: Genera
LONDON  — 
Former FBI
0 19 2 0 A
Chris Blac
Following 
26 Shares

(Adult Lan
Departing 
The   at t
Today, in 
By now, ev
Donald J. 
Monday on 
The unrave
CNN host D
PHILADELPH
WASHINGTON
There is a
Nina Falco
CNN is fea
Thursday, 
Since it s
Gordie How
Trump VP’s
The April 
Fall and R
The world 
The brain 
Next Prev 
The main C
  Disgrace
We Are Cha
The Mexica
Kaboom! Me
A new vide
De e-mails
 
OTTAWA  — 
The dead m
  mothers.
Club Soda 
By Cassius
Thursday o
It is inte
WASHINGTON
45 Views N
0 44 
To b
by Brian S
Brexiters 
FMs of Ira
AUSTIN, Te
Women shou
Why Americ
0 comments
David Broo
Man at din
Singer and
Comic book
While roll
  Crooked 
While ther
The Truth 
2 Replies 
PALERMO, I
.@SenSan

100%|███████████████████████████████████████████████████████████████████████████| 20652/20652 [00:25<00:00, 805.72it/s]

After Sund
Trump Towe
HOUSTON  —
Share on F
Lady Gaga 
Leave a re
Wednesday 
After thre
Email 
As 
By Vigilan
Like most 
Google Map
 
In 2002,
WASHINGTON
During his
by Yves Sm
Leonard Li
California
Europe Hop
Email 
As 
SNOWFLAKE:
With more 
Secretary 
Videos Rig
TEL AVIV  
Donald J. 
CORAL GABL
BERNARDS T
Joseph Soh
Americans 
WASHINGTON
Регион: СШ
The Office
(Reuters) 
Poll: Trum
Imported g
adobochron
10 Strange
They are  
With the U
PARIS, Fra
The Daily 
Email 
Not
Comments 

An ethics 
Audio-Slid
FRISCO, Te
The tweets
(Want to g
The March 
A car thie
Sex Bombsh
Republican
President-
Savory tar
Objetivos:
On the Wed
Videos US 
Nine Other
This Is Th
Home | Wor
18 mins ag
MONTECITO,
Posted on 
The Wall S
What if Bi
November 4
AUGUSTA, G
Judges are
Shocking s
LONDON  — 
Several th
Pentagon L
  Donald T
Ryan McMak
The Politi
Nomination
Breitbart.
In these t
Home | Wor
Freitag, 2
Like me, y
If Jared K
WASHINGTON
By Michael
Thomas Fra
WASHINGTON
On Monday’
Fuck you! 
Migrant Cr
Bill Clint

In [66]:
prepped_data['full_text_clean'] = prepped_data['full_text'].progress_map(clean_text)

  1%|▍                                                                            | 129/20652 [00:00<00:31, 646.12it/s]

House Dem 
FLYNN: Hil
Why the Tr
15 Civilia
Iranian wo
Jackie Mas
Life: Life
Benoît Ham
Excerpts F
A Back-Cha
Obama’s Or
BBC Comedy
Russian Re
US Officia
Re: Yes, T
In Major L
Wells Farg
Anonymous 
FBI Closes
Chuck Todd
News: Hope
Monica Lew
Rob Reiner
Massachuse
Abortion P
Nukes and 
EXCLUSIVE:
Humiliated
Andrea Tan
How Hillar
Chuck Todd
Israel is 
Having Won
Texas Oil 
Bayer Deal
Russia Mov
Re: Why We
Open Threa
Democrat G
Avoiding P
MRI Shows 
4 of the B
Ryan Locht
Can I have
Conservati
Internal I
Press TV D
СМИ Сербии
Samsung, A
Poland Vow
Sparking A
Dueling Dr
Study: Mor
 Sounds li
Samantha B
The Trump 
Ep. 544 FA
Cognition 
If Donald 
Mindful Ea
The Major 
I wonder w
3 Makers o
Massive An
Review: ‘L
U.S. Gener
Jury finds
Clinton Ca
Pence Will
Bernie San
How To Mak
Treason! N
Dress Like
At 91, Ell
Pressing A
Democrats 
News: PR D
Judge span
51 U.S. Di
Franken Ca
Louisiana,
Turkey Thr
Huma’s Wei
Colin Kaep
Trump’s Im
Mary Tyler
Poison By 
Trump Fans
Fox Biz Re
11 Fiction
Mike Birbi

  1%|▉                                                                            | 266/20652 [00:00<00:30, 663.44it/s]

Closed Afg
TV Anchors
Pelosi: Re
The Beauti
I Ignored 
Donald Tru
EU, Finlan
FBI Direct
Montana De
The Monsan
The Shame,
It’s Offic
 There is 
It Literal
U.N. Secre
Trump Boll
FBI Finds 
2 Years Af
Report: Il
Make Nethe
Four kille
The Leader
Students A
Despite St
The Rise o
A Newly Vi
Fed Holds 
The Battle
The Latest
Burlesque 
What the C
For Cuomo 
The Top 10
New Study 
James Matt
A Black Ch
Sears Agre
Takata Chi
Goodbye, f
Teen ’Geis
Mohamad Kh
Remember T
Price on O
VA fails t
Trump fami
Sports Wri
Spider-pig
‘I Can Wat
This Open 
Comment on
Teacher un
Hijacking 
You Don’t 
Scientists
Pro-Govern
WHO cancer
Work. Walk
Militants 
Steve Harv
Coalition:
UK citizen
After Vets
Mr. Trump’
Here Is Ho
20 Foods T
Death of t
Watch: Pol
 They got 
Comment on
Could Teac
Donald Tru
Miami Beac
Doctors My
Donald Tru
Shocking N
The Failur
EXCLUSIVE 
Trump Tell
Will Ameri
Commission
Re: Babylo
The UN Pla
Trump Atta
Clinton Ad
Art Laffer
How Donald
New World 
Pokemon Go
California
Exclusive:
In Break W

  2%|█▌                                                                           | 418/20652 [00:00<00:28, 699.94it/s]

...

Iran to Ba
Here's The
Home but N
Doomsday C
Trump Thro
Donald Tru
The Politi
Jordan Fre
Palme d’Or
Monitoring
Slovenia A
Biggest El
Von wegen 
Angry Town
Obamacare 
Democrats 
THE END GA
Sandy Hook
Texas Dem 
Places to 
Lower Yiel
Travel Ban
UP IN ARMS
A Call For
Comment on
Donald Tru
Trump: Gov
Sparkling 
Alabama Go
Megyn Kell
’Hot Mugsh
Social Med
Trump Coup
Amazon Is 
WARNING IO
BIGGER Qua
Blue State
DeepStateG
Michelle O
Mormons’ D
Trump Advo
‘UFO’ Spot
 I hope no
Oklahoma M
EU Politic
An Alterna
World’s Ho
‘Duck Dyna
Silicon Va
Portugal D
Hillary Cl
On a Desig
The Secret
For Those 
Woman Resc
Watch: The
ESPN’s His
James Come
40-Year Se
 Another e
How Do You
Why isnt o
Exposed: A
Satisfacti
Why Can’t 
Hidden Cam
Second Ave
Women Susp
“Got trick
The Adults
Investors 
Students D
Walter Hau
AMERICA CE
17 Colombi
Generation
Thousands 
Uber Defie
Bob Dylan’
NATO, US, 
Soldier He
A Traditio
YouTube Re
If You Dre
Farage to 
Американо-
Who Is Ele
Homeschool
Coal Miner
Podesta Em
WikiLeaks:

 99%|██████████████████████████████████████████████████████████████████████████▍| 20514/20652 [00:25<00:00, 797.45it/s]

Donald Tru
Barbra Str
Clinton's 
CNN’s Don 
Donald Tru
Fact Check
Which Powe
Credit Car
CNN Featur
Biden and 
Arianna Hu
Gordie How
Trump VP’s
Lena Dunha
Fall and R
Global Tem
Updated Br
A very acc
Egyptian C
Disgraced 
LEAKED AUD
In Wake of
Alien Visi
Video Show
De e-mails
SAID IN SP
Quebec Mos
Learning t
‘What if Y
Club Soda 
Wikileaks 
Conway: Me
 It is int
Shadow of 
With Gold 
White 0 44
FREE Summi
Brexiters 
FMs of Ira
How Apple 
Let Women 
Why Americ
WATCH How 
David Broo
Man at din
Nick Canno
Marvel Art
President 
Crooked Hi
The Met Op
The Truth 
Put Some M
Scaling Up
Sanders: ’
 Hyperbole
New to Air
NYU Prof W
21 Things 
Novak Djok
Beyond The
The #Never
Pakistan: 
Fresno Sta
Hackers De
REPENT!!! 
As Anthem-
Jewish Pir
Reanimatio
Game Of Th
UN: Migran
Кто любит 
With Preet
Drone Rest
Virgil: Ma
Trump: ’We
Here’s Why
Men Have B
‘All Talk,
Trump will
Will Trump
 Amusing c
America at
MI5: Bonds
FNC’s Gera
Comment on
Comment on
Donald Tru
Desperate 
Google Acc
Re: Will T
Re: Don’t 

100%|███████████████████████████████████████████████████████████████████████████| 20652/20652 [00:25<00:00, 806.16it/s]

Comment on
‘Moonlight
Trump Towe
Federal Co
The Truth 
Lady Gaga 
The Sleep 
America th
Battle Ove
Tom Hayden
BBC’s Chil
The Week A
How to Use
13 Years L
Kellyanne 
Trump: ’I 
How Your S
Leonard Li
California
Europe Hop
Former Att
SNOWFLAKE:
Hillary Cl
WATCH: Sec
Right-Wing
WATCH: Van
Donald Tru
ACLU Launc
The Mosque
The Pathol
A Push to 
On Hillary
Способен л
That’s Not
Putin Says
Poll: Trum
Four Illeg
Comment on
Comment on
Outsiders 
How to Hav
Exclusive:
Leaked Ema
Top Radio 
Kim Davis 
Ethics Com
Kunst am W
Tony Romo’
As America
OPEC, Dona
Pro-life G
Car Thief 
Sex Bombsh
Celebritie
Foreign Po
A Vegetabl
¿Quién es 
Breitbart 
US Calls O
US Airstri
This Is Th
Experts: O
Donald Tru
Sperm Bank
Comment on
Wall Stree
Lessons in
For our fr
Dustin Joh
DHS Office
Shock Vide
Cyprus: Wh
PICS: Thou
Pentagon L
Donald Tru
Trade Agre
The Politi
Art-House 
Treason: C
Jackie Mas
Vive la Ré
Kleiner Vo
The Red Pi
It Begins!
Trump to S
The Battle
Thomas Fra
For Paul R
Brzezinski
Fuck you! 
Why Did Fo

In [67]:
display(prepped_data[['text','text_clean','full_text', 'full_text_clean']].head(10))

,text,text_clean,full_text,full_text_clean
0,House Dem Aide: We Didn’t Even See Comey’s Let...,house dem aide we did not even see comey s let...,House Dem Aide: We Didn’t Even See Comey’s Let...,house dem aide we did not even see comey s let...
1,Ever get the feeling your life circles the rou...,ever get the feeling your life circles the rou...,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",flynn hillary clinton big woman on campus brei...
2,"Why the Truth Might Get You Fired October 29, ...",why the truth might get you fired october 29 2...,Why the Truth Might Get You Fired Why the Trut...,why the truth might get you fired why the trut...
3,Videos 15 Civilians Killed In Single US Airstr...,videos 15 civilians killed in single us airstr...,15 Civilians Killed In Single US Airstrike Hav...,15 civilians killed in single us airstrike hav...
4,Print \nAn Iranian woman has been sentenced to...,print an iranian woman has been sentenced to s...,Iranian woman jailed for fictional unpublished...,iranian woman jailed for fictional unpublished...
5,"In these trying times, Jackie Mason is the Voi...",in these trying times jackie mason is the voic...,Jackie Mason: Hollywood Would Love Trump if He...,jackie mason hollywood would love trump if he ...
6,Ever wonder how Britain’s most iconic pop pian...,ever wonder how britain s most iconic pop pian...,Life: Life Of Luxury: Elton John’s 6 Favorite ...,life life of luxury elton john s 6 favorite sh...
7,"PARIS — France chose an idealistic, traditi...",paris france chose an idealistic traditional c...,Benoît Hamon Wins French Socialist Party’s Pre...,beno t hamon wins french socialist party s pre...
8,Donald J. Trump is scheduled to make a highly ...,donald j trump is scheduled to make a highly a...,Excerpts From a Draft Script for Donald Trump’...,excerpts from a draft script for donald trump ...
9,A week before Michael T. Flynn resigned as nat...,a week before michael t flynn resigned as nati...,"A Back-Channel Plan for Ukraine and Russia, Co...",a back channel plan for ukraine and russia cou...


In [68]:
def tokenize_and_lemmatize(s):
    print(s[:10])
    tokens = word_tokenize(s)
    tokens = [token for token in tokens 
              if token in ENGLISH_WORDS 
              and token not in STOP_WORDS
              and len(token)>2]
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    return tokens

In [ ]:
prepped_data['tokens'] = prepped_data['text_clean'].progress_map(tokenize_and_lemmatize)

In [ ]:
prepped_data['full_tokens'] = prepped_data['full_text_clean'].progress_map(tokenize_and_lemmatize)

In [71]:
prepped_data[['text', 'tokens', 'full_text', 'full_tokens']].head(10)

,text,tokens,full_text,full_tokens
0,House Dem Aide: We Didn’t Even See Comey’s Let...,"[house, aide, even, see, letter, jason, darrel...",House Dem Aide: We Didn’t Even See Comey’s Let...,"[house, aide, even, see, letter, jason, house,..."
1,Ever get the feeling your life circles the rou...,"[ever, get, feeling, life, roundabout, rather,...","FLYNN: Hillary Clinton, Big Woman on Campus - ...","[hillary, clinton, big, woman, campus, ever, g..."
2,"Why the Truth Might Get You Fired October 29, ...","[truth, might, get, fired, october, tension, i...",Why the Truth Might Get You Fired Why the Trut...,"[truth, might, get, fired, truth, might, get, ..."
3,Videos 15 Civilians Killed In Single US Airstr...,"[single, rate, american, higher, engaged, acti...",15 Civilians Killed In Single US Airstrike Hav...,"[single, single, rate, american, higher, engag..."
4,Print \nAn Iranian woman has been sentenced to...,"[print, iranian, woman, six, prison, iran, rev...",Iranian woman jailed for fictional unpublished...,"[iranian, woman, fictional, unpublished, story..."
5,"In these trying times, Jackie Mason is the Voi...","[trying, time, mason, voice, reason, week, exc...",Jackie Mason: Hollywood Would Love Trump if He...,"[mason, hollywood, would, love, trump, bombed,..."
6,Ever wonder how Britain’s most iconic pop pian...,"[ever, wonder, britain, iconic, pop, pianist, ...",Life: Life Of Luxury: Elton John’s 6 Favorite ...,"[life, life, luxury, john, favorite, shark, st..."
7,"PARIS — France chose an idealistic, traditi...","[paris, chose, idealistic, traditional, candid...",Benoît Hamon Wins French Socialist Party’s Pre...,"[beno, french, socialist, party, presidential,..."
8,Donald J. Trump is scheduled to make a highly ...,"[donald, trump, make, highly, visit, church, s...",Excerpts From a Draft Script for Donald Trump’...,"[draft, script, donald, trump, black, church, ..."
9,A week before Michael T. Flynn resigned as nat...,"[week, michael, resigned, national, security, ...","A Back-Channel Plan for Ukraine and Russia, Co...","[back, channel, plan, russia, courtesy, trump,..."


In [74]:
prepped_data['text_lemmatized'] = prepped_data['tokens'].map(lambda tokens: " ".join(tokens))

In [75]:
prepped_data['full_text_lemmatized'] = prepped_data['full_tokens'].map(lambda tokens: " ".join(tokens))

In [72]:
prepped_data['token_count'] = prepped_data['tokens'].map(len)

In [73]:
prepped_data['full_token_count'] = prepped_data['full_tokens'].map(len)

In [76]:
prepped_data.head()

,title,text,label,full_text,char_count,text_clean,full_text_clean,tokens,full_tokens,token_count,full_token_count,text_lemmatized,full_text_lemmatized
0,House Dem Aide: We Didn’t Even See Comey’s Let...,House Dem Aide: We Didn’t Even See Comey’s Let...,1,House Dem Aide: We Didn’t Even See Comey’s Let...,5012,house dem aide we did not even see comey s let...,house dem aide we did not even see comey s let...,"[house, aide, even, see, letter, jason, darrel...","[house, aide, even, see, letter, jason, house,...",320,326,house aide even see letter jason darrell octob...,house aide even see letter jason house aide ev...
1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Ever get the feeling your life circles the rou...,0,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",4216,ever get the feeling your life circles the rou...,flynn hillary clinton big woman on campus brei...,"[ever, get, feeling, life, roundabout, rather,...","[hillary, clinton, big, woman, campus, ever, g...",287,292,ever get feeling life roundabout rather straig...,hillary clinton big woman campus ever get feel...
2,Why the Truth Might Get You Fired,"Why the Truth Might Get You Fired October 29, ...",1,Why the Truth Might Get You Fired Why the Trut...,7726,why the truth might get you fired october 29 2...,why the truth might get you fired why the trut...,"[truth, might, get, fired, october, tension, i...","[truth, might, get, fired, truth, might, get, ...",476,480,truth might get fired october tension intellig...,truth might get fired truth might get fired oc...
3,15 Civilians Killed In Single US Airstrike Hav...,Videos 15 Civilians Killed In Single US Airstr...,1,15 Civilians Killed In Single US Airstrike Hav...,3301,videos 15 civilians killed in single us airstr...,15 civilians killed in single us airstrike hav...,"[single, rate, american, higher, engaged, acti...","[single, single, rate, american, higher, engag...",192,193,single rate american higher engaged active com...,single single rate american higher engaged act...
4,Iranian woman jailed for fictional unpublished...,Print \nAn Iranian woman has been sentenced to...,1,Iranian woman jailed for fictional unpublished...,1032,print an iranian woman has been sentenced to s...,iranian woman jailed for fictional unpublished...,"[print, iranian, woman, six, prison, iran, rev...","[iranian, woman, fictional, unpublished, story...",62,71,print iranian woman six prison iran revolution...,iranian woman fictional unpublished story woma...


### Deleting Records with no valid tokens

In [77]:
prepped_data.describe()

,label,char_count,token_count,full_token_count
count,20652.000000,20652.000000,20652.000000,20652.000000
mean,0.497046,4628.469543,298.989057,304.664149
std,0.500003,5129.856542,335.125542,335.515437
min,0.000000,2.000000,0.000000,0.000000
25%,0.000000,1712.750000,106.000000,111.000000
50%,0.000000,3444.500000,220.000000,226.000000
75%,1.000000,6359.000000,415.000000,421.000000
max,1.000000,143035.000000,9279.000000,9284.000000


In [78]:
prepped_data.loc[prepped_data['full_token_count']==0]

,title,text,label,full_text,char_count,text_clean,full_text_clean,tokens,full_tokens,token_count,full_token_count,text_lemmatized,full_text_lemmatized
47,"СМИ Сербии приписали россиянам ""подготовку тер...",0 комментариев 0 поделились Фото: AP \nОднако ...,1,"СМИ Сербии приписали россиянам ""подготовку тер...",3651,0 0 ap n1 26 27 16 16 20 viber whatsapp 27 27,0 0 ap n1 26 27 16 16 20 viber whatsapp 27 27,[],[],0,0,,
371,"Путин рассказал, когда в Крыму решат проблему ...",0 комментариев 7 поделились \nОтвечая на соотв...,1,"Путин рассказал, когда в Крыму решат проблему ...",2013,0 7 23 2015 2020 40 4 5 2 5 25,0 7 23 2015 2020 40 4 5 2 5 25,[],[],0,0,,
492,Казахстан на страже ядерной безопасности | Нов...,В ноябре 2016 г. Мажилис Парламента Республики...,1,Казахстан на страже ядерной безопасности | Нов...,6525,2016 2006 2011 2015 2015 2016 3 2016 2016 2018...,2016 2006 2011 2015 2015 2016 3 2016 2016 2018...,[],[],0,0,,
580,NaN,Ludicrous...,1,Ludicrous...,13,,,[],[],0,0,,
650,Очередная автоколонна МЧС с гуманитарной помощ...,19 МЧС направило 57-ю по счёту автоколонну с ...,1,Очередная автоколонна МЧС с гуманитарной помощ...,757,19 57 40 440 2014 56 64,19 57 40 440 2014 56 64,[],[],0,0,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...
18556,Минобороны России: Эвакуация жителей Алеппо бы...,"\nКонашенков пояснил, что представителями си...",1,Минобороны России: Эвакуация жителей Алеппо бы...,2188,40 afp 2011 2011 220 2014 17,40 afp 2011 2011 220 2014 17,[],[],0,0,,
19009,Опрос: Россияне одобряют действия президента П...,0 комментариев 4 поделились Фото: AP \nКак пок...,1,Опрос: Россияне одобряют действия президента П...,2768,0 4 ap 78 7 81 8 53 6 58 3 39 7 45 5 42 1 46 4...,0 4 ap 78 7 81 8 53 6 58 3 39 7 45 5 42 1 46 4...,[],[],0,0,,
19734,Москва ввела в эксплуатацию систему по борьбе ...,"\nКак пишет Коммерсант, система под название...",1,Москва ввела в эксплуатацию систему по борьбе ...,2801,5 50 934 60 26 100 36 volkswagen 12 36 4 06 00...,5 50 934 60 26 100 36 volkswagen 12 36 4 06 00...,[],[],0,0,,
20168,Стала известна возможная причина взрыва дома в...,Фото: © Пресс-служба МЧС по Рязанской области ...,1,Стала известна возможная причина взрыва дома в...,2309,23 10 1 279 61 162 31 pravda ru,23 10 1 279 61 162 31 pravda ru,[],[],0,0,,


In [79]:
prepped_data = prepped_data.drop(index=prepped_data.loc[prepped_data['full_token_count']==0].index)
prepped_data.shape

(20546, 13)

In [80]:
prepped_data.describe()

,label,char_count,token_count,full_token_count
count,20546.000000,20546.000000,20546.000000,20546.000000
mean,0.494451,4639.159252,300.531588,306.235958
std,0.499981,5138.287902,335.298338,335.663579
min,0.000000,5.000000,0.000000,1.000000
25%,0.000000,1716.000000,107.000000,112.000000
50%,0.000000,3467.000000,222.000000,227.000000
75%,1.000000,6370.750000,416.000000,422.000000
max,1.000000,143035.000000,9279.000000,9284.000000


In [81]:
prepped_data.to_csv('data/preprocessed_data.csv', index=False)

In [2]:
prepped_data = pd.read_csv('data/preprocessed_data.csv')

In [3]:
prepped_data[['full_text', 'full_tokens', 'full_text_lemmatized', 'label']].to_csv('data/fulltext_preprocessed_data.csv', index=False)

In [11]:
prepped_data2 = prepped_data[['text', 'tokens', 'text_lemmatized', 'label']]
prepped_data2 = prepped_data2.drop(index=prepped_data[prepped_data['token_count']==0].index)
prepped_data2.shape

(20459, 4)

In [12]:
prepped_data.to_csv('data/text_preprocessed_data.csv', index=False)